# 🤖 Intelligent Chatbot using Hugging Face Transformers

### 👩‍💻 Developed by: Karri Dhanusha

### Data Science Internship – February 2026

This project presents the implementation of a conversational chatbot using a pre-trained transformer model from Hugging Face. The chatbot interacts with users and generates meaningful responses dynamically.

## 🎯 Objective

The objective of this assignment is to build a chatbot using transformer-based models that can:
- Understand user input
- Generate intelligent responses
- Maintain a conversational flow

## 🛠️ Tools and Technologies Used

- Python
- Hugging Face Transformers
- PyTorch
- Google Colab

## 📦 Libraries Used

- transformers → For loading pre-trained NLP models  
- torch → For tensor operations and model execution  

!pip install transformers torch

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

## 🤖 Why DialoGPT?

DialoGPT is chosen because it is specifically designed for conversational tasks. It is based on the GPT architecture and is fine-tuned on dialogue datasets, making it suitable for building chatbots.

Compared to general models like GPT-2, DialoGPT performs better in maintaining conversational context.

## 🤖 Model Selection

In this project, we use **DialoGPT-medium**, a pre-trained transformer model from Hugging Face designed specifically for conversational AI tasks.

It generates human-like responses and maintains conversation context effectively.

In [2]:
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set model to evaluation mode (important for inference)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)

## 💬 Chatbot Implementation

The chatbot works in the following steps:

1. Accept user input
2. Tokenize user input into a model-understandable format
3. Generate response using transformer-based language modeling
4. Display output
5. Repeat until user exits

In [8]:
# Chatbot logic (Improved Version)

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day 😊")
        break

    # Encode input with attention mask
    new_input = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
    new_input_ids = new_input["input_ids"]
    attention_mask = new_input["attention_mask"]

    # Maintain chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        attention_mask = torch.cat([torch.ones(chat_history_ids.shape, dtype=torch.long), attention_mask], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_new_tokens=100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    # Decode response
    bot_output = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", bot_output)

Chatbot: Hello! I am your AI assistant. How can I help you today?
You: Hello
Chatbot: I hope you have a wonderful day!
You: What is Artificial Intelligence?
Chatbot: It's the thing that allows us to play games and things. It's a thing, right?
You: What is Machine Learning?
Chatbot: A way of life.
You: Who created Python?
Chatbot: You didn't know about python until the early 2000s...
You: exit
Chatbot: Goodbye! Have a great day 😊


## ⚠️ Note on Model Responses

The responses generated by the chatbot may sometimes appear less accurate or generic. This is because the model used (DialoGPT-medium) is a pre-trained conversational model and is not fine-tuned for specific factual accuracy.

However, the objective of this project is to demonstrate the implementation of transformer-based conversational AI and dynamic response generation.

## ⚠️ Limitations

- The model is not fine-tuned for factual accuracy  
- Responses may sometimes be incorrect or irrelevant  
- Limited long-term memory (context resets over time)  
- Cannot handle domain-specific queries effectively  

These limitations can be improved using fine-tuning and external knowledge integration.

- No external knowledge base integration (pure generative model)

## ⚙️ Model Parameters Explanation

- **max_new_tokens**: Limits response length
- **temperature**: Controls randomness (lower = focused, higher = creative)
- **top_k**: Selects top probable words
- **top_p**: Controls diversity using cumulative probability
- **repetition_penalty**: Avoids repeated text
- **no_repeat_ngram_size**: Prevents repeating phrases

These parameters help improve response quality and make the chatbot more natural.

## 📊 Sample Output (Actual Behavior)

You: What is Artificial Intelligence?  
Chatbot: A big ol'ole robot.

You: What is Machine Learning?  
Chatbot: That's not a real thing...

⚠️ Note: Responses may vary and may not always be factually correct.

## 🧠 Model Architecture (Advanced Explanation)

DialoGPT is based on the Transformer architecture, which uses self-attention mechanisms to understand context in conversations.

It follows causal language modeling, where each token is generated based on previous tokens, enabling coherent and context-aware responses.

## ⚙️ How It Works

The chatbot uses a transformer-based model (DialoGPT) which processes user input and generates responses based on context.

Pipeline:
User Input → Tokenization → Context Integration → Transformer Processing → Probabilistic Response Generation → Output Display

This pipeline represents the flow of data through the transformer-based architecture.

## 📌 Key Features of the Chatbot

- Real-time conversational interaction  
- Context-aware responses using chat history  
- Controlled response generation using sampling techniques  
- Scalable design for integration into applications  

## 🌍 Real-World Applications

- Customer Support Chatbots
- Virtual Assistants
- Educational Q&A Systems
- Healthcare Chat Assistants
- E-commerce Recommendation Bots

## 🚀 Future Improvements

- Integrate chatbot with a web interface (Gradio/Streamlit)
- Improve response quality using fine-tuned models
- Add memory for long conversations
- Deploy chatbot as a web application

✅ Conclusion

In this project, we successfully built a conversational chatbot using a transformer-based model (DialoGPT). The chatbot demonstrates the ability to generate dynamic, context-aware responses using natural language processing techniques.

This implementation highlights how transformer architectures enable scalable and intelligent conversational systems. It also provides a strong foundation for building real-world AI applications such as virtual assistants, customer support systems, and domain-specific chatbots.